# InterPro Fetch

![InterPro Fetch](https://proto-bio.github.io/proto-assets/images/tool/interproscan/hero.png)

This notebook demonstrates `run_interproscan_fetch`, which retrieves protein domain annotations from [InterPro](https://www.ebi.ac.uk/interpro/) — the EMBL-EBI aggregator covering Pfam, SMART, PROSITE, Gene3D / CATH-Gene3D, Panther, and the rest of the InterPro member-database catalog.

Two paths share the same Output schema: a fast **direct lookup** by UniProt accession (no submission required), and a **submit-and-scan** path against EBI's iprscan5 service for novel sequences (requires a contact email per EBI policy).

In [1]:
from proto_tools.tools.database_retrieval import (
    InterProScanFetchConfig,
    InterProScanFetchInput,
    run_interproscan_fetch,
)
from proto_tools.utils.notebook_docs import display_api_reference

## Input API Reference

Provide exactly one of `uniprot_id` or `sequence`.

In [2]:
display_api_reference("interproscan", "input")

**Input** — `InterProScanFetchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>uniprot_id</code> | <code>str &#124; None</code> | <code>None</code> | UniProt accession for direct entry lookup |
| <code>sequence</code> | <code>str &#124; None</code> | <code>None</code> | Protein sequence for submit-and-scan path |

## Config API Reference

In [3]:
display_api_reference("interproscan", "config")

**Config** — `InterProScanFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>email</code> | <code>str &#124; None</code> | <code>None</code> | EBI contact email for the sequence-submit path. Defaults to the INTERPROSCAN_EMAIL env var. |
| <code>applications</code> | <code>list[Literal['PfamA', 'Panther', 'Gene3d', 'SuperFamily', 'SMART', 'PrositeProfiles', 'PrositePatterns', 'PRINTS', 'PIRSF', 'FunFam', 'HAMAP', 'CDD', 'NCBIfam', 'SFLD', 'Coils', 'MobiDBLite', 'Phobius', 'SignalP', 'SignalP_EUK', 'SignalP_GRAM_POSITIVE', 'SignalP_GRAM_NEGATIVE', 'AntiFam', 'PIRSR', 'TMHMM']] &#124; None</code> | <code>None</code> | Submit-only — restrict to subset of InterPro member DBs; None runs the EBI default set |
| <code>include_go_terms</code> | <code>bool</code> | <code>True</code> | Include GO term cross-references in the output |
| <code>include_pathways</code> | <code>bool</code> | <code>True</code> | Reactome/KEGG/MetaCyc xrefs on the sequence path; no-op on UniProt-id path |
| <code>sequence_type</code> | <code>Literal['protein', 'nucleic']</code> | <code>'protein'</code> | Sequence-submit path: 'protein' or 'nucleic' (6-frame translated server-side) |

## Output API Reference

In [4]:
display_api_reference("interproscan", "output")

**Output** — `InterProScanFetchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>accession</code> | <code>str &#124; None</code> | <code>None</code> | Resolved UniProt accession |
| <code>sequence_length</code> | <code>int &#124; None</code> | <code>None</code> | Queried protein length |
| <code>domains</code> | <code>list[InterProDomain]</code> | <code>[]</code> | InterPro hits across all member DBs |
| <code>num_domains</code> | <code>int</code> | required | Total number of hits |
| <code>job_id</code> | <code>str</code> | required | iprscan5 job ID for the sequence path (empty for direct lookup) |
| <code>source_url</code> | <code>str</code> | required | InterPro entry / iprscan5 result URL |
| <code>raw_entries</code> | <code>list[dict[str, Any]]</code> | <code>[]</code> | Raw API JSON entries |

**`InterProDomain`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>accession</code> | <code>str</code> | required | Member-DB accession (PfamID, IPR-ID, G3DSA-ID, ...) |
| <code>name</code> | <code>str</code> | required | Human-readable domain/family name |
| <code>type</code> | <code>Literal['family', 'domain', 'repeat', 'active_site', 'conserved_site', 'homologous_superfamily', 'binding_site', 'ptm', 'unknown']</code> | required | Category (family, domain, repeat, ...) |
| <code>member_database</code> | <code>str</code> | required | Source DB ('pfam', 'panther', 'cathgene3d', ...) |
| <code>integrated_ipr</code> | <code>str &#124; None</code> | <code>None</code> | Parent InterPro accession; None for non-integrated member-DB hits |
| <code>start</code> | <code>int</code> | required | 1-indexed inclusive start residue |
| <code>end</code> | <code>int</code> | required | 1-indexed inclusive end residue |
| <code>score</code> | <code>float &#124; None</code> | <code>None</code> | Per-database score: e-value (smaller is better) for HMM DBs, otherwise bit score (larger is better) |
| <code>model</code> | <code>str &#124; None</code> | <code>None</code> | HMM/profile model ID |
| <code>representative</code> | <code>bool</code> | <code>False</code> | Whether this is the representative match |
| <code>go_terms</code> | <code>list[str]</code> | <code>[]</code> | GO term IDs cross-referenced from this entry |
| <code>pathways</code> | <code>list[str]</code> | <code>[]</code> | Pathway IDs (Reactome, MetaCyc, ...) |

Each `InterProDomain` row carries `accession`, `name`, `type`, `member_database`, `integrated_ipr`, 1-indexed inclusive `start` / `end`, optional `score` / `model` / `representative`, and `go_terms` / `pathways` lists.

## Basic Usage

Fetch all InterPro hits for human TP53 (UniProt `P04637`) via the fast direct path. No contact email required because no job is submitted.

In [5]:
output = run_interproscan_fetch(InterProScanFetchInput(uniprot_id="P04637"))
assert output.success
print(f"{output.num_domains} hits across {sorted({d.member_database for d in output.domains})}")

27 hits across ['cathgene3d', 'cdd', 'interpro', 'panther', 'pfam', 'prints', 'prosite', 'ssf']


## Example: Find the canonical Pfam DNA-binding-domain anchor for TP53

In [6]:
for domain in output.domains:
    if domain.member_database == "pfam" and domain.name.startswith("P53"):
        print(f"{domain.accession} {domain.name}: {domain.start}-{domain.end} ({domain.type})")

PF00870 P53 DNA-binding domain: 100-288 (domain)
PF07710 P53 tetramerisation motif: 319-357 (conserved_site)
PF08563 P53 transactivation motif: 6-30 (conserved_site)


## Example: Build a `lock_mask` for "preserve the active site, redesign the rest"

Locks every residue covered by an `active_site`, `conserved_site`, `binding_site`, or `ptm` match. Use the resulting boolean mask as a per-residue constraint for any redesign tool.

In [7]:
length = output.sequence_length or max(d.end for d in output.domains)
lock_mask = [False] * length
for domain in output.domains:
    if domain.type in {"active_site", "conserved_site", "binding_site", "ptm"}:
        for i in range(domain.start - 1, domain.end):
            lock_mask[i] = True
print(f"{sum(lock_mask)} of {length} residues locked")

102 of 393 residues locked


## Submit-and-scan path (novel sequences)

When you have a sequence that is not in UniProt, submit it to iprscan5. **Set `config.email` to a real contact mailbox** — EBI requires it. Wall-clock varies (~2–30 min) with queue depth; the helper polls automatically.

```python
output = run_interproscan_fetch(
    InterProScanFetchInput(sequence="MKTI...your sequence..."),
    InterProScanFetchConfig(email="you@example.org"),
)
```